In [1]:
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
import math
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import johnsonsu
from scipy.optimize import minimize
import csv
import time

#1. Getting prices
def prices(startd, endd, tic):
    info = []
    key = '&apikey=ZKMMTO1ATDBLXH2K'
    ticker = '&symbol=' + str(tic)
    endpoint = 'function=TIME_SERIES_DAILY_ADJUSTED'
    size = '&outputsize=full'
    web = 'https://www.alphavantage.co/query?'
    url = web + endpoint + ticker + size + key

    r = requests.get(url)
    if r.status_code == 200:
        print('connection successful')
        data = r.json()
        r1 = data.get('Time Series (Daily)', {})
        for date, values in sorted(r1.items()):
            if startd <= date <= endd:
                info.append([tic, date, values['5. adjusted close']])
    return info


def compute_log_returns(data, freq='daily'):
    """
    Computes log returns at specified frequency.

    Parameters:
    -----------
    data : list of lists
        Price data: [[ticker, date, adjusted_close], ...]
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'

    Returns:
    --------
    list of lists: [[ticker, date, adjusted_close, log_return], ...]
    """
    sorted_data = sorted(data, key=lambda x: x[1])
    data_with_logret = []

    if freq == 'daily':
        for i in range(len(sorted_data)):
            if i == 0:
                data_with_logret.append(sorted_data[i] + [None])
            else:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[i-1][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])

    elif freq == 'weekly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 7:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i

    elif freq == 'monthly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 30:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i

    elif freq == 'yearly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 360:
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i

    return data_with_logret


def download_and_export(ticker, startd, endd, freq='daily', path=None):
    """
    Downloads price data, computes log and simple returns, and exports to CSV.

    Parameters:
    -----------
    ticker : str
        Stock ticker symbol (e.g. 'NVDA')
    startd : str
        Start date 'YYYY-MM-DD'
    endd : str
        End date 'YYYY-MM-DD'
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'
    path : str
        File path for CSV output. Defaults to '{ticker}_returns.csv'

    Returns:
    --------
    list of lists: [[ticker, date, price, log_return, simple_return], ...]
    First row is a header row.
    """
    if path is None:
        path = f"{ticker}_returns.csv"

    raw = prices(startd, endd, ticker)
    with_log = compute_log_returns(raw, freq=freq)
    with_both = compute_returns(with_log)

    with open(path, mode='w', newline='') as f:
        writer = csv.writer(f)
        for row in with_both:
            writer.writerow(row)

    print(f"Exported {len(with_both)-1} rows to {path}")
    return with_both

def days_30_360(start_date, end_date):
    """
    Calculate the number of days between two dates using the 30/360 US convention.
    Dates expected as 'YYYY-MM-DD' strings or datetime.date-like objects.

    Returns an integer day-count.
    """
    if isinstance(start_date, str):
        d1 = datetime.strptime(start_date, '%Y-%m-%d').date()
    else:
        d1 = start_date
    if isinstance(end_date, str):
        d2 = datetime.strptime(end_date, '%Y-%m-%d').date()
    else:
        d2 = end_date
    d1_day = min(d1.day, 30)
    # Adjust d2 day per 30/360 US rule
    if d2.day == 31 and d1_day == 30:
        d2_day = 30
    else:
        d2_day = min(d2.day, 30)
    return 360 * (d2.year - d1.year) + 30 * (d2.month - d1.month) + (d2_day - d1_day)


In [2]:
# ===== ALL INPUTS =====
tickers     = ["JPM", "AAPL", "MSFT", "GS"]
market      = "SPY"
start_date  = "2000-01-01"   # full download range
end_date    = "2026-01-01"
rf          = 0.05            # annual risk-free rate
rf_monthly  = rf / 12
cash_pct    = 0.05            # cash allocation %
I           = 10000           # initial investment
long_fee    = 0.0001
short_fee   = 0.0001

start_dates = [
    "2010-01-01","2016-01-01","2020-01-01"
]


In [3]:
def prices_month(startd, tic):
    info = []
    key = '&apikey=ZKMMTO1ATDBLXH2K'
    ticker = '&symbol=' + str(tic)
    endpoint = 'function=TIME_SERIES_MONTHLY_ADJUSTED'
    size = '&outputsize=full'
    web = 'https://www.alphavantage.co/query?'
    url = web + endpoint + ticker + size + key

    r = requests.get(url)
    if r.status_code == 200:
        print('connection successful')
        data = r.json()
        r1 = data.get('Monthly Adjusted Time Series', {})
        for date, values in sorted(r1.items()):
            if date >= startd:
                info.append([tic, date, values['5. adjusted close']])
    return info


def monthly_log_returns(startd, tic):
    raw = prices_month(startd, tic)
    return compute_log_returns(raw, freq='daily')


def sigma_matrix(tickers, startd, window=60):
    series_ret = {}
    series_price = {}
    for i, tic in enumerate(tickers):
        if i > 0:
            time.sleep(3)
        data = monthly_log_returns(startd, tic)
        rows = [row for row in data if row[3] is not None]
        series_ret[tic]   = pd.Series({row[1]: row[3]        for row in rows})
        series_price[tic] = pd.Series({row[1]: float(row[2]) for row in rows})

    returns_df = pd.DataFrame(series_ret).dropna()
    prices_df  = pd.DataFrame(series_price).reindex(returns_df.index)
    df_train   = returns_df.iloc[:window]
    return df_train.cov(), returns_df, prices_df


def calculate_p0_p1_p2(prices_df, tickers):
    p0 = prices_df[tickers].iloc[60].values.reshape(-1, 1)
    p1 = prices_df[tickers].iloc[61].values.reshape(-1, 1)
    p2 = prices_df[tickers].iloc[62].values.reshape(-1, 1)
    p0 = np.vstack([p0, [[1.0]]])
    p1 = np.vstack([p1, [[1.0]]])
    p2 = np.vstack([p2, [[1.0]]])
    return p0, p1, p2


In [6]:
# Download all data (run once)
sigma, returns_df, prices_df = sigma_matrix(tickers + [market], start_date)

# --- Step 1: Monthly Sigma (first 60 returns) ---
print("Monthly Sigma Matrix:")
display(sigma.drop(market).drop(market, axis=1))

# --- Step 2: Beta ---
var_market = sigma.loc[market, market]
betas = sigma[market].drop(market) / var_market
print("\nBetas vs SPY:")
display(betas)

# --- Step 3: Month-62 realized market risk premium ---
R_market_62 = returns_df.iloc[60][market]
market_risk_premium = R_market_62 - rf_monthly
print(f"\nSPY return at month 62:         {R_market_62:.6f}")
print(f"Market risk premium (month 62): {market_risk_premium:.6f}")

# --- Step 4: CAPM expected monthly returns ---
mu = rf_monthly + betas * market_risk_premium
print("\nCAPM Expected Monthly Returns (for implementation at month 62):")
display(mu)


connection successful
connection successful
connection successful
connection successful
connection successful
Monthly Sigma Matrix:


,JPM,AAPL,MSFT,GS
JPM,0.012441,0.007704,0.005695,0.007405
AAPL,0.007704,0.035497,0.009890,0.010177
MSFT,0.005695,0.009890,0.014708,0.005057
GS,0.007405,0.010177,0.005057,0.011090



Betas vs SPY:


JPM     1.822798
AAPL    1.980926
MSFT    1.521774
GS      1.597605
Name: SPY, dtype: float64


SPY return at month 62:         0.020689
Market risk premium (month 62): 0.016522

CAPM Expected Monthly Returns (for implementation at month 62):


JPM     0.034283
AAPL    0.036895
MSFT    0.029309
GS      0.030562
Name: SPY, dtype: float64

In [7]:
# Tangent portfolio — CAPM mu, monthly sigma
sigma_assets = sigma.drop(market).drop(market, axis=1)
excess = mu.values - rf_monthly
sigma_inv = np.linalg.inv(sigma_assets.values)

z = sigma_inv @ excess
T = z / z.sum()

tangent = pd.Series(T, index=mu.index)
print("Tangent Portfolio Weights:")
display(tangent)

tangent_with_cash = pd.concat([tangent * (1 - cash_pct), pd.Series([cash_pct], index=['Cash'])])
print("\nWeights with Cash:")
display(tangent_with_cash)


Tangent Portfolio Weights:


JPM     0.435911
AAPL    0.039193
MSFT    0.225677
GS      0.299218
dtype: float64


Weights with Cash:


JPM     0.414116
AAPL    0.037233
MSFT    0.214394
GS      0.284257
Cash    0.050000
dtype: float64

In [8]:
def max_drawdown(e0, e_p1, e_p2):
    equity = np.array([np.sum(e0), np.sum(e_p1), np.sum(e_p2)])
    peak   = equity[0]
    mdd    = 0.0
    for val in equity[1:]:
        if val > peak:
            peak = val
        dd = (peak - val) / peak
        if dd > mdd:
            mdd = dd
    print("Equity path: {}".format(equity.round(2)))
    print("Max Drawdown: {:.4f} ({:.2f}%)".format(mdd, mdd * 100))
    return mdd


In [9]:
def initial_values(tangent_w, p0, k0, I):
    w0 = tangent_w.values.reshape(-1, 1)
    n = len(w0.ravel())

    sign = np.sign(w0).ravel()
    D = np.diag(sign / k0.ravel())
    ones = np.ones((1, n))
    zero = np.array([[0]])

    A = np.block([[D, -1*w0], [ones, zero]])
    C = np.zeros((n + 1, 1))
    C[-1, 0] = I

    m = np.linalg.inv(A) @ C
    m0 = m[0:n].round(3)

    if np.sum(m0) > I + (I/100) or np.sum(m0) < I - (I/100):
        print("Margin Allocations != I")

    e0  = m0
    a0  = (e0/k0) * np.sign(w0)
    l0  = np.where(w0>0, a0-e0, a0*-1)
    s0  = a0 / p0
    er0 = k0
    unlocked_cash = float(a0[-1])
    c0  = np.array([unlocked_cash, 0.0])
    f0  = np.zeros((n, 1))

    print("----------  Initial Values  ----------")
    print("# Shares",    s0.ravel().round(3))
    print("$ Assets",    a0.ravel().round(3))
    print("$ Equity",    e0.ravel().round(3))
    print("$ Liability", l0.ravel().round(3))
    print("E Ratio",     er0.ravel().round(3))
    print("$ Margin",    m0.ravel().round(3))
    print(f"Unlocked C {c0[0]} - Locked C {c0[1]}")

    return s0, a0, e0, l0, er0, m0, c0, f0

In [10]:
def price_change(p0, p1, s, a, e, l, er, m, c, f, long_fee, short_fee):
    a1 = p1 * s
    l1 = np.where(a1>0, l, l + (p0-p1)*s)
    e1 = np.where(a1>0, e + (a1 - a), e - (p0-p1)*s)
    er1 = e1 / a1 * np.sign(a1)

    todays_long_fee  = np.where(a>0, l, 0) * long_fee
    todays_short_fee = np.where(a<0, l, 0) * short_fee
    todays_total_fee = todays_long_fee + todays_short_fee
    e1 = e1 - todays_total_fee
    f1 = f + todays_total_fee

    e1[-1] = e[-1]  # cash equity unchanged by price moves

    print("----------  After Price Change  ----------")
    print("# Shares",    s.ravel().round(3))
    print("$ Assets",    a1.ravel().round(3))
    print("$ Equity",    e1.ravel().round(3))
    print("$ Liability", l1.ravel().round(3))
    print("E Ratio",     er1.ravel().round(3))
    print("$ Margin",    m.ravel().round(3))
    print("$ Fees",      f1.ravel().round(3))
    print(f"Unlocked C {c[0]} - Locked C {c[1]}")

    return s, a1, e1, l1, er1, m, c, f1

In [11]:
def margin_call(s, a, e, l, er, m, c, f):
    ulcash = c[0]
    lcash  = c[1]

    mc   = np.where(er < 0.25, "YES", "NO")
    en   = 0.25 * a * np.sign(a)                    # minimum equity needed per position
    eadd = np.where(er < 0.25, en - e, 0)           # equity to add per position

    print("Equity Right Now");       print(e.ravel().round(3))
    print("Total Equity Required");  print(en.ravel().round(3))
    print("Equity Need to Add");     print(eadd.ravel().round(3))

    if np.sum(eadd) > 0:
        print("Need Margin Call")
        lcash  += np.sum(eadd)
        e       = e + eadd
        ulcash -= np.sum(eadd)
        m       = m + eadd

        if ulcash < 0:
            cadd = -1 * ulcash
            print(f"We need to add ${float(np.asarray(cadd).squeeze()):.2f} of cash")
            ulcash = 0
        else:
            print("Did not need to add more cash")

        c = np.array([float(np.asarray(ulcash).squeeze()),
                      float(np.asarray(lcash).squeeze())], dtype=float)
    else:
        print("No margin call needed")

    er = e / a * np.sign(a)

    print("\n----------  After Margin Call  ----------")
    print("# Shares",    s.ravel().round(3))
    print("$ Assets",    a.ravel().round(3))
    print("$ Equity",    e.ravel().round(3))
    print("$ Liability", l.ravel().round(3))
    print("E Ratio",     er.ravel().round(3))
    print("$ Margin",    m.ravel().round(3))
    print(f"Unlocked C {round(c[0], 2)} - Locked C {round(c[1], 2)}")

    return s, a, e, l, er, m, c, f


In [12]:
def rebalance2(p, s, a, e, l, er, m, c, f, tangent_w):
    w0 = tangent_w.values.reshape(-1, 1)

    w1   = a / a.sum()
    dist = np.linalg.norm(w1 - w0) / np.linalg.norm(w0)

    if dist > 0.05:
        print(f"Distance is {dist} YES rebalance (rebalance2).")

        V      = np.sum(a)
        a_star = w0 * V
        d      = a_star - a
        a1     = a + d

        l1 = np.where(a1>0, l + ((a1 - a) * (1 - er)), np.abs(a1))
        e1 = np.where(a1>0, e + ((a1 - a) * (er)),      a1 * er)
        e1 = np.abs(e1)
        s1 = a1 / p

        delta_s   = s1 - s
        rebal_val = delta_s * p
        m1 = np.abs(np.where(w1>0, m + rebal_val * er, m - rebal_val * er))

        # cash position
        delta_cash = float(a1[-1] - a[-1])
        ulcash     = float(c[0]) + delta_cash
        lcash      = float(c[1])
        m1[-1]     = m[-1] + delta_cash
        e1[-1]     = e[-1] + delta_cash
        l1[-1]     = 0.0
        c1         = np.array([ulcash, lcash])
    else:
        a1 = a; e1 = e; s1 = s; l1 = l; m1 = m; c1 = c
        print(f"Distance is {dist} NO rebalance.")

    print("\n----------  After Rebalance  ----------")
    print("# Shares",    s1.ravel().round(3))
    print("$ Assets",    a1.ravel().round(3))
    print("$ Equity",    e1.ravel().round(3))
    print("$ Liability", l1.ravel().round(3))
    print("E Ratio",     er.ravel().round(3))
    print("$ Margin",    m1.ravel().round(3))
    print(f"Unlocked C {round(c1[0], 2)} - Locked C {round(c1[1], 2)}")

    return s1, a1, e1, l1, er, m1, c1, f

In [5]:
def sharpe_ratio(e0, e_p1, e_p2, rf_monthly):
    """
    Sharpe ratio over two monthly periods using portfolio-level ROE.
    e0:   equity at implementation (from initial_values)
    e_p1: equity after period 1 rebalance
    e_p2: equity after period 2 rebalance
    """
    roe1 = (np.sum(e_p1) - np.sum(e0))  / np.sum(e0)
    roe2 = (np.sum(e_p2) - np.sum(e_p1)) / np.sum(e_p1)

    returns = np.array([roe1, roe2])
    excess  = returns - rf_monthly
    sharpe  = (np.mean(returns)-rf_monthly) / np.std(returns)

    print(f"ROE Period 1: {roe1:.6f}")
    print(f"ROE Period 2: {roe2:.6f}")
    print(f"Mean Excess Return: {np.mean(excess):.6f}")
    print(f"Std of Returns:     {np.std(returns, ddof=1):.6f}")
    print(f"Sharpe Ratio:       {sharpe:.4f}")

    return sharpe

In [13]:
def max_drawdown(e0, e_p1, e_p2):
    equity = np.array([np.sum(e0), np.sum(e_p1), np.sum(e_p2)])
    peak   = equity[0]
    mdd    = 0.0
    for val in equity[1:]:
        if val > peak:
            peak = val
        dd = (peak - val) / peak
        if dd > mdd:
            mdd = dd
    print('Equity path: {}'.format(equity.round(2)))
    print('Max Drawdown: {:.4f} ({:.2f}%)'.format(mdd, mdd * 100))
    return mdd


In [14]:
def run_pipeline(returns_df, prices_df, window_start, tickers, market, rf, rf_monthly,
                 cash_pct, I, long_fee, short_fee, window=60):
    train  = returns_df.iloc[window_start : window_start + window]
    sigma  = train.cov()
    var_market = sigma.loc[market, market]
    betas      = sigma[market].drop(market) / var_market
    R_market_62         = returns_df.iloc[window_start + window][market]
    market_risk_premium = R_market_62 - rf_monthly
    mu = rf_monthly + betas * market_risk_premium
    sigma_assets = sigma.drop(market).drop(market, axis=1)
    excess_ret   = mu.values - rf_monthly
    sigma_inv    = np.linalg.inv(sigma_assets.values)
    z = sigma_inv @ excess_ret
    T = z / z.sum()
    tangent           = pd.Series(T, index=mu.index)
    tangent_with_cash = pd.concat([tangent * (1 - cash_pct),
                                   pd.Series([cash_pct], index=["Cash"])])
    p0 = np.vstack([prices_df[tickers].iloc[window_start + 60].values.reshape(-1, 1), [[1.0]]])
    p1 = np.vstack([prices_df[tickers].iloc[window_start + 61].values.reshape(-1, 1), [[1.0]]])
    p2 = np.vstack([prices_df[tickers].iloc[window_start + 62].values.reshape(-1, 1), [[1.0]]])
    k0 = np.vstack([np.full((len(tangent), 1), 0.5), [[1.0]]])
    s0, a0, e0, l0, er0, m0, c0, f0 = initial_values(tangent_with_cash, p0, k0, I)
    s, a, e, l, er, m, c, f = price_change(p0, p1, s0, a0, e0, l0, er0, m0, c0, f0, long_fee, short_fee)
    s, a, e, l, er, m, c, f = margin_call(s, a, e, l, er, m, c, f)
    s, a, e, l, er, m, c, f = rebalance2(p1, s, a, e, l, er, m, c, f, tangent_with_cash)
    e_p1 = e.copy()
    s, a, e, l, er, m, c, f = price_change(p1, p2, s, a, e, l, er, m, c, f, long_fee, short_fee)
    s, a, e, l, er, m, c, f = margin_call(s, a, e, l, er, m, c, f)
    s, a, e, l, er, m, c, f = rebalance2(p2, s, a, e, l, er, m, c, f, tangent_with_cash)
    e_p2 = e.copy()
    sr  = sharpe_ratio(e0, e_p1, e_p2, rf_monthly)
    mdd = max_drawdown(e0, e_p1, e_p2)
    return {"tangent_with_cash": tangent_with_cash, "mu": mu, "betas": betas,
            "s0": s0, "a0": a0, "e0": e0, "l0": l0, "er0": er0, "m0": m0, "c0": c0,
            "s":  s,  "a":  a,  "e":  e,  "l":  l,  "er":  er,  "m":  m,  "c":  c,  "f": f,
            "e_p1": e_p1, "e_p2": e_p2, "sharpe": sr,"mdd":mdd,
            "p0": p0, "p1": p1, "p2": p2,
            "date_p0": prices_df.index[window_start + 60]}

In [15]:
sharpe_vector = np.zeros(len(start_dates))
mdd_vector    = np.zeros(len(start_dates))
results       = []

for i, sd in enumerate(start_dates):
    window_start = returns_df.index.searchsorted(sd)
    print(window_start)
    print()
    print('='*50)
    print('Window {}/{}  |  start: {}  |  P0: {}'.format(
        i+1, len(start_dates), sd, returns_df.index[window_start + 60]))
    print('='*50)
    result = run_pipeline(
        returns_df, prices_df, window_start,
        tickers, market, rf, rf_monthly,
        cash_pct, I, long_fee, short_fee
    )
    sharpe_vector[i] = result['sharpe']
    mdd_vector[i]    = result['mdd']
    results.append(result)

print()
print('='*50)
print('Results across all windows:')
print('{:<10} {:<14} {:<10} {}'.format('Window', 'P0 Date', 'Sharpe', 'Max DD'))
for i, (r, sr, mdd) in enumerate(zip(results, sharpe_vector, mdd_vector)):
    print('{:<10} {:<14} {:<10.4f} {:.4f}'.format(i+1, str(r['date_p0']), sr, mdd))


119

Window 1/3  |  start: 2010-01-01  |  P0: 2015-01-30
----------  Initial Values  ----------
# Shares [165.526 142.429 134.404  22.602 952.381]
$ Assets [6662.856 3695.95  4617.85  3118.582  952.381]
$ Equity [3331.428 1847.975 2308.925 1559.291  952.381]
$ Liability [3331.428 1847.975 2308.925 1559.291    0.   ]
E Ratio [0.5 0.5 0.5 0.5 1. ]
$ Margin [3331.428 1847.975 2308.925 1559.291  952.381]
Unlocked C 952.381 - Locked C 0.0
----------  After Price Change  ----------
# Shares [165.526 142.429 134.404  22.602 952.381]
$ Assets [7508.278 4068.302 5047.85  3443.673  952.381]
$ Equity [4176.517 2220.143 2738.694 1884.226  952.381]
$ Liability [3331.428 1847.975 2308.925 1559.291    0.   ]
E Ratio [0.556 0.546 0.543 0.547 1.   ]
$ Margin [3331.428 1847.975 2308.925 1559.291  952.381]
$ Fees [0.333 0.185 0.231 0.156 0.   ]
Unlocked C 952.381 - Locked C 0.0
Equity Right Now
[4176.517 2220.143 2738.694 1884.226  952.381]
Total Equity Required
[1877.07  1017.076 1261.962  860.918  238.

/var/folders/18/j7dymdq94xn2rz2v9p514qqm0000gn/T/ipykernel_39841/1521042624.py:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  unlocked_cash = float(a0[-1])
/var/folders/18/j7dymdq94xn2rz2v9p514qqm0000gn/T/ipykernel_39841/1521042624.py:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  unlocked_cash = float(a0[-1])
/var/folders/18/j7dymdq94xn2rz2v9p514qqm0000gn/T/ipykernel_39841/1521042624.py:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  unlocked_cash = float(

In [ ]:
p0, p1, p2 = calculate_p0_p1_p2(prices_df, tickers)

k0 = np.vstack([np.full((len(tangent), 1), 0.5), [[1.0]]])

s0, a0, e0, l0, er0, m0, c0, f0 = initial_values(tangent_with_cash, p0, k0, I)


In [ ]:
s, a, e, l, er, m, c, f = price_change(p0, p1, s0, a0, e0, l0, er0, m0, c0, f0, long_fee, short_fee)


In [ ]:
s, a, e, l, er, m, c, f = margin_call(s, a, e, l, er, m, c, f)

In [ ]:
s, a, e, l, er, m, c, f = rebalance2(p1, s, a, e, l, er, m, c, f, tangent_with_cash)
e_p1 = e.copy()

In [ ]:
s, a, e, l, er, m, c, f = price_change(p1, p2, s0, a0, e0, l0, er0, m0, c0, f0, long_fee, short_fee)

In [ ]:
s, a, e, l, er, m, c, f = margin_call(s, a, e, l, er, m, c, f)


In [ ]:
s, a, e, l, er, m, c, f = rebalance2(p2, s, a, e, l, er, m, c, f, tangent_with_cash)
e_p2 = e.copy()

In [ ]:
sharpe = sharpe_ratio(e0, e_p1, e_p2, rf_monthly)

In [ ]:
max_drawdown(e0, e_p1, e_p2)